In [1]:
from platform import python_version
print(python_version())

3.10.19


## Samtools and bcftools

### About BCFtools

BCFtools is a program for variant calling and manipulating files in the Variant Call Format (VCF) and its binary counterpart BCF. All commands work transparently with both VCFs and BCFs, both uncompressed and BGZF-compressed. In order to avoid tedious repetition, throughout this document we will use "VCF" and "BCF" interchangeably, unless specifically noted.

Most commands accept VCF, bgzipped VCF and BCF with filetype detected automatically even when streaming from a pipe. Indexed VCF and BCF work in all situations. Unindexed VCF and BCF and streams work in most, but not all situations. In general, whenever multiple VCFs are read simultaneously, they must be indexed and therefore also compressed.

https://samtools.github.io/bcftools/howtos/index.html

https://github.com/samtools/bcftools

### About GATK

[GATK](https://gatk.broadinstitute.org/hc/en-us)

### About Mustect2

[Mutect2](https://gatk.broadinstitute.org/hc/en-us/articles/360037593851-Mutect2)

In [2]:
import os, sys, yaml
from pathlib import Path
from dotenv import load_dotenv

import numpy as np
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

from scipy.stats import spearmanr

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

ROOT0 = Path("/home/flavio/uv/mutation_analysis")
ROOT_SRC = ROOT0 / "src"

sys.path.insert(0, str(ROOT_SRC))

if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import create_dir, pdreadcsv, pdwritecsv

root_figure = create_dir(ROOT0, "pictures")
root_results = create_dir(ROOT0, "results")
root_data = create_dir(ROOT0, "data")
root_vcf = create_dir(root_data, "vcf")

from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

ROOT0: /home/flavio/uv/mutation_analysis
ROOT_SRC added: /home/flavio/uv/mutation_analysis/src


I'll put the design and what to measure in this reply, and write the commands into three files: a runnable-but-unexecuted pipeline script, a sample sheet with the FASTQ layout, and a small Python helper for the clonality input. Nothing will be downloaded or run.Everything is in the three files. `bash -n` and `py_compile` pass. The script only prints a notice unless it is run with `--run`, so it downloads nothing and runs nothing.

### a) Pipeline design

I've assumed these are WGS reads (FASTQ from the three tissues, CRAM from GTEx).

```
FASTQ (T, B, N) ─┐
GTEx CRAM → FASTQ┴▶ BWA-MEM2 → MarkDuplicates → BQSR → QC (depth, contamination, somalier identity)
                                                    │
        ┌───────────────────────┬───────────────────┼──────────────────────┬─────────────────┐
   Germline (N+GTEx)      PoN (GTEx only)     Somatic, per patient     SV / CNV          TEs, mtDNA
   HaplotypeCaller GVCF   Mutect2 → PoN       Mutect2 joint T+B vs N   Manta, Delly      MELT, xTea
   → joint genotyping                         + Strelka2 (2nd caller)  → SURVIVOR         Mutect2 chrM
                                              → FilterMutectCalls      ASCAT purity/CN
                                                    │
                                     VEP → PyClone-VI (T+B CCF) → signatures, MSI
```

#### **What each sample does in the design:**

| Sample | Role | Watch out for |
|---|---|---|
| **N (non-tumour)** | Matched germline control for somatic calling | Tumour cells or field-effect clones in N. If N carries tumour mutations, Mutect2 rejects them as germline. Estimate this with **TINC**. If it is high, use blood as the normal. |
| **B (border)** | Tells you whether the invasion front, field cancerization or pre-malignant clones are present | Low purity. Sequence deeper and expect ASCAT to fail; the helper script has `--purity` to override it. |
| **T (tumour)** | Primary somatic profile | Purity, subclonality, and whole-genome doubling |
| **GTEx** | Panel of normals, plus population germline background | **GTEx WGS comes from blood, not from the tissue**, and from a different sequencing batch. It is not a matched normal and it can't show tissue-level mosaicism. Access requires dbGaP (phs000424). |

#### **Key design choices:**
- **Realign everything from FASTQ** with the same aligner version, including GTEx. Otherwise batch artefacts show up as "somatic" variants and in the panel of normals.
- **Joint multi-sample Mutect2 (T + B, normal N).** This gives read counts for every variant in both T and B, even where one sample has zero alt reads. Clonality needs exactly that.
- **Build the PoN from GTEx, not from N.** Adjacent normal may contain tumour hotspot mutations, which would then be filtered from every patient.
- **Depth:** T ≥ 60–80×, B ≥ 80–100×, N ≥ 30–40×. Low-VAF signal in B and N needs the extra depth.

#### b) What to measure

Ordered by how much each tells you with this design:

1. **Somatic SNVs and indels.** Drivers, TMB, and above all *sharing*: each mutation classified as T-only, T+B, or T+B+N.
2. **Copy number, purity, ploidy, LOH, whole-genome doubling** (ASCAT). Clonality needs these, and they matter on their own.
3. **Clonal architecture ("clones").** Cancer cell fraction per mutation in T and B, clusters, and a phylogeny (e.g., CONIPHER or PairTree after PyClone-VI). This is the central question: does B carry the truncal tumour clone (infiltration), only earlier clones (field cancerization), or its own independent clones?
4. **Structural variants.** Deletions, amplifications, translocations, chromothripsis, fusions. The script uses a consensus of at least 2 callers.
5. **Mutational signatures.** SBS, ID and CN signatures per sample. HRD can be scored with HRDetect or CHORD on top of these. Comparing signatures between T and B shows which processes act early versus late.
6. **MSI** (msisensor-pro).
7. **Germline predisposition** in N, e.g. HR and MMR genes, checked against GTEx and gnomAD frequencies.
8. **Transposable elements.** Somatic L1, Alu and SVA insertions (xTea) are common in some epithelial tumours and can mark clonal lineages. Germline MEIs (MELT) are mostly context. Lower priority.
9. **mtDNA heteroplasmy and copy number.** Cheap lineage markers shared between T and B.
10. **Optional extras from the same BAMs:** telomere length (TelomereHunter) and viral integration (VIRUSBreakend), if the tumour type makes it relevant.

WGS doesn't give you expression, methylation or the immune microenvironment. For those you'd need RNA-seq or methylation data from the same regions.

#### c) Files

- **samplesheet.csv** has one row per lane, with columns `patient, sex, status, sample, lane, fastq_1, fastq_2`. Samples follow the naming `<PT>_T/_B/_N` and `GTEX-*`. It uses the same format as nf-core/sarek.
- **pipeline.sh** has one function per step (0–13) and a `main` that loops over the sample sheet. The header also gives an nf-core/sarek command that covers steps 1–10 if you'd rather use a managed workflow.
- **make_pyclone_input.py** combines the joint Mutect2 VCF with ASCAT segments and purity into PyClone-VI input (cyvcf2 + pandas).

Four things you'll still need to adapt before running:
- The xTea call is left as a comment, because its flags change between releases.
- Germline VQSR or hard filtering after `GenotypeGVCFs` is noted but not written out.
- Strelka2 output isn't intersected with Mutect2 yet. A consensus with a "2 of 2 callers" rule is the usual approach.
- The `ref/` resource paths assume the Broad GRCh38 bundle.

In [3]:
[x for x in os.listdir(root_data) ]

['samplesheet_prostate.csv', 'clinical_prostate.csv', 'vcf', 'samplesheet.csv']

In [4]:
fname = 'samplesheet.csv'

df = pdreadcsv(fname, root_data, sep=',')
df

,patient,sex,status,sample,lane,fastq_1,fastq_2
0,P01,XX,1,P01_T,L001,fastq/P01_T_S1_L001_R1_001.fastq.gz,fastq/P01_T_S1_L001_R2_001.fastq.gz
1,P01,XX,1,P01_T,L002,fastq/P01_T_S1_L002_R1_001.fastq.gz,fastq/P01_T_S1_L002_R2_001.fastq.gz
2,P01,XX,1,P01_B,L001,fastq/P01_B_S2_L001_R1_001.fastq.gz,fastq/P01_B_S2_L001_R2_001.fastq.gz
3,P01,XX,1,P01_B,L002,fastq/P01_B_S2_L002_R1_001.fastq.gz,fastq/P01_B_S2_L002_R2_001.fastq.gz
4,P01,XX,0,P01_N,L001,fastq/P01_N_S3_L001_R1_001.fastq.gz,fastq/P01_N_S3_L001_R2_001.fastq.gz
5,P02,XY,1,P02_T,L001,fastq/P02_T_S4_L001_R1_001.fastq.gz,fastq/P02_T_S4_L001_R2_001.fastq.gz
6,P02,XY,1,P02_B,L001,fastq/P02_B_S5_L001_R1_001.fastq.gz,fastq/P02_B_S5_L001_R2_001.fastq.gz
7,P02,XY,0,P02_N,L001,fastq/P02_N_S6_L001_R1_001.fastq.gz,fastq/P02_N_S6_L001_R2_001.fastq.gz
8,GTEX-DONOR01,XY,0,GTEX-DONOR01,L001,fastq/GTEX-DONOR01_L001_R1.fastq.gz,fastq/GTEX-DONOR01_L001_R2.fastq.gz
9,GTEX-DONOR02,XX,0,GTEX-DONOR02,L001,fastq/GTEX-DONOR02_L001_R1.fastq.gz,fastq/GTEX-DONOR02_L001_R2.fastq.gz


In [5]:
fname = 'samplesheet_prostate.csv'
dfp = pdreadcsv(fname, root_data, sep=',')
dfp

,patient,cohort,group,sample,bam
0,PR01,cancer,T,PR01_T1,ega/PR01_T1.bam
1,PR01,cancer,T,PR01_T2,ega/PR01_T2.bam
2,PR01,cancer,B,PR01_B,ega/PR01_B.bam
3,PR01,cancer,N,PR01_N,ega/PR01_N.bam
4,PR02,cancer,T,PR02_T1,ega/PR02_T1.bam
5,PR02,cancer,B,PR02_B,ega/PR02_B.bam
6,PR02,cancer,N,PR02_N,ega/PR02_N.bam
7,HP01,healthy,H,HP01_H,ega/HP01_H.bam
8,HP01,healthy,N,HP01_N,ega/HP01_N.bam
9,HP02,healthy,H,HP02_H,ega/HP02_H.bam


In [6]:
fname = 'clinical_prostate.csv'
dfc = pdreadcsv(fname, root_data, sep=',')
dfc


,patient,cohort,isup_grade_group,gleason,pT,pN,M
0,PR01,cancer,NaN,NaN,NaN,NaN,NaN
1,PR02,cancer,NaN,NaN,NaN,NaN,NaN
2,HP01,healthy,NaN,NaN,NaN,NaN,NaN
3,HP02,healthy,NaN,NaN,NaN,NaN,NaN


### CRUK-ICGC - Prostate Cancer

Prostate it is. The CRUK-ICGC data differ from the generic pipeline in five ways, so I'll build a dedicated version rather than patch the old one:

- **BAM input.** Reads come as EGA BAMs, which need reverting and realigning.
- **Multifocal tumours.** Patients have several tumour regions each (38 tumours in 30 men), not a single T.
- **Healthy-prostate arm.** The 7 cancer-free men (prostate plus blood) replace GTEx.
- **Clinical filter.** ISUP grade group 2–3 and M0, applied from a clinical sheet.
- **Prostate driver report.** TMPRSS2–ERG, SPOP, FOXA1, PTEN, and so on.Small fix: exclude healthy samples from ASCAT by group, not by name suffix.`pipeline_prostate.py` is set up for the CRUK-ICGC prostate data (EGAD00001004125 + EGAD00001000689). I dry-ran it on the example sheet (2 cancer patients, 2 healthy men). It printed about 400 commands and the clinical filter behaved as intended: patients with missing clinical data are excluded, a grade group 4 / pN1 case is dropped, and a grade group 2 / M0 case is kept. Nothing was downloaded or executed. `make_pyclone_input.py` is unchanged and still required.

**Sample groups (one row per sample in `samplesheet_prostate.csv`):**

| Group | Sample | Role |
|---|---|---|
| **T** | Tumour region, `<PT>_T1`, `_T2`, … | Multifocal tumours, all called jointly per patient |
| **B** | Morphologically normal prostate from the same patient | Border / field tissue |
| **H** | Prostate from a cancer-free man | Healthy-tissue baseline. Replaces GTEx. |
| **N** | Blood | Germline control for everyone |

**Main design changes from the generic pipeline:**
- **Input is EGA BAM.** It is reverted per read group with RevertSam, then realigned to GRCh38. Original lane and library tags are kept, so duplicate marking stays correct. Optical duplicate distance is set to 100 for HiSeq 2000.
- **One joint Mutect2 per man.** For cancer patients it covers all tumour regions plus B against blood; for healthy men, H against blood. The output shows directly which mutations are private to one tumour, shared between tumours, or shared with the "normal" prostate.
- **PoN from blood only** (all 37 men, same platform). Prostate tissue is kept out on purpose: clonal expansions shared across prostates are the field-effect signal you want to measure, not noise to filter.
- **The key comparison is B vs H.** The mutation burden, clone sizes and signatures in morphologically normal tissue next to cancer are compared with healthy prostates of similar age. That separates field cancerization from ordinary age-related mosaicism.
- **Clinical filter.** Patients are kept only with ISUP grade group 2–3 and M0. `--pN0` also requires node-negative. Missing clinical data excludes a patient unless you pass `--allow-missing-clinical`.
- **Prostate driver report.**
  - Coding hits in SPOP, FOXA1, PTEN, TP53, CHD1, NKX3-1, the HR/MMR genes, and others.
  - A TMPRSS2–ERG check on the chr21 deletion/breakpoint window, for every tissue sample.
- **ASCAT** runs on tumour regions and B, not on H. MSI runs on tumours only.
- **Mitochondrial workflow** runs on every sample. The recurrent-artefact list comes from the 37 unrelated bloods.
- **`--steps fetch`** prints the `pyega3` download commands and does nothing else.

**Before running:**
1. **Apply for EGA access** through DAC EGAC00001000010.
2. **Fill in `samplesheet_prostate.csv`.** After download, use the EGA metadata to map each BAM to its man and group. The current names are placeholders.
3. **Fill in `clinical_prostate.csv`** (grade group, Gleason, pT, pN, M) from the metadata that comes with the access. It is all `NA` now, and I couldn't confirm per-patient grades publicly, so the final number of eligible men is unknown until you have it.
4. **Handle low border purity in clonality.** Normal tissue was sequenced at about 30×, so low-VAF clones in B will be hard to detect. If ASCAT fails on B, reuse a tumour region's copy-number segments with an explicit purity, as described in the comment in `clonality()`.

### CRUK-ICGC - Prostate Cancer - wget + credentials

The public part works with `wget`, but the BAMs themselves can't be fetched that way: they're controlled access, and EGA only serves them to its authenticated client, `pyega3`, after the ICGC DAC approves your request. `fetch_cruk_prostate.sh` covers both parts. I haven't run it: this workspace can't reach EGA from the shell. The syntax check passes and the parsing works on real records I pulled from the API.

**Public metadata, no login needed:**
```bash
API=https://metadata.ega-archive.org
wget -q -O files.json   "$API/datasets/EGAD00001004125/files?limit=100&offset=0"
wget -q -O samples.json "$API/datasets/EGAD00001004125/samples?limit=100&offset=0"
wget -q -O icgc_dac.json "$API/dacs/EGAC00001000010/datasets"
```
`./fetch_cruk_prostate.sh metadata` does this for both datasets, goes through every page, and writes TSVs (file ID, size, MD5; sample ID, patient, tissue label). It also prints the file count and TB per dataset, so you can check storage before downloading about 8 TB.

**BAMs, after approval:**
```bash
pip install pyega3
echo '{"username":"you@inst.org","password":"..."}' > ega_credentials.json && chmod 600 ega_credentials.json
pyega3 -cf ega_credentials.json files EGAD00001004125                     # lists file names
pyega3 -cf ega_credentials.json -c 8 fetch EGAF00002010748 --output-dir ega
```
`./fetch_cruk_prostate.sh download [ids.txt]` fetches either every file or only the IDs you list. `verify` re-checks each MD5 against the public metadata.

What the metadata already shows:
- **Sample labels carry the patient and tissue**, e.g. `0002_CRUK_PC_0002_Blood` with patient `0002_CRUK_PC_0002`. That's enough to start filling `samplesheet_prostate.csv`.
- **Some samples have two entries from different sequencing centres.** Patient 0001's `_N` sample appears once as BGI and once as Illumina. Decide whether to merge or pick one; don't count them as two tissues.
- **Check what `_N` means before assigning groups.** It could be normal prostate or blood, and a label like that is exactly what decides B versus N.
- **The public API doesn't link files to samples.** File names, which include sample IDs, only appear through `pyega3 files` once you're authorised. Download a subset (e.g. grade group 2–3 patients) only after you have that mapping.
- **Files are large:** the first three are 85–106 GB each.

Sources:
- [EGA public metadata API](https://ega-archive.org/discovery/metadata/public-metadata-api/)
- [EGA download client (pyega3)](https://github.com/EGA-archive/ega-download-client)
- [EGA dataset EGAD00001004125](https://ega-archive.org/datasets/EGAD00001004125)
- [EGA dataset EGAD00001000689](https://ega-archive.org/datasets/EGAD00001000689)

You don't need an AWS account for this. The ICGC bucket is public, so `--no-sign-request` skips credentials completely.

#### Linux (x86_64), official AWS CLI v2

```bash
curl "https://awscli.amazonaws.com/awscli-exe-linux-x86_64.zip" -o awscliv2.zip
unzip awscliv2.zip
sudo ./aws/install
aws --version
```

On ARM (Graviton, some clusters), use `awscli-exe-linux-aarch64.zip` instead.

**Without sudo** (HPC or a shared server):
```bash
./aws/install -i ~/.local/aws-cli -b ~/.local/bin
export PATH="$HOME/.local/bin:$PATH"   # add this line to ~/.bashrc
```

#### Python alternative (uv or pip)

This installs AWS CLI v1 from PyPI. It's fine for `ls` and `cp` on a public bucket.

```bash
uv tool install awscli        # or: pip install --user awscli
aws --version
```

#### macOS and Windows

- **macOS:** `brew install awscli`
- **Windows:** `msiexec.exe /i https://awscli.amazonaws.com/AWSCLIV2.msi`

#### Using it on the ICGC bucket

```bash
EP=https://object.genomeinformatics.org
aws s3 ls --no-sign-request --endpoint-url $EP s3://icgc25k-open/
```

The bucket isn't hosted on AWS, so the region value doesn't matter. If you get "You must specify a region", add `--region us-east-1`.

### I dont have credentials

That's fine, as long as you're affiliated with an institution: EGA credentials come with ICGC data-access approval, and you can apply for it. There are also two things you can download today without any credentials.

#### Getting access to the CRUK prostate BAMs

Access is managed by ICGC's data access office (DACO), which now runs on the ICGC ARGO platform.
1. **Apply at [daco.icgc-argo.org](https://docs.icgc-argo.org/docs/data-access/daco/applying)** with a Google-linked email.
   - The PI must be an independent researcher at a legal entity.
   - The application needs the PI's signature, a signature from an **institutional representative** who can legally commit the institution (usually the research office), and ethics approval if your country requires it.
   - Expect weeks, not days.
2. **After approval, log in to EGA with your institutional email.** Access to the datasets under EGAC00001000010 appears [within about 48 h](https://docs.icgc-argo.org/docs/data-access/icgc-25k-data).
3. **Then `pyega3` and `fetch_cruk_prostate.sh download` work** as written.

#### 1. Open ICGC data: check grades before you apply

The legacy ICGC 25K open release is in a public bucket and needs no credentials. The CRUK prostate project is `PRAD-UK`. The open release includes the clinical and specimen tables (where tumour grade should be recorded) and the somatic calls, but not the reads. With it you can count how many patients are ISUP grade group 2–3 and M0 before investing in the application.

```bash
EP=https://object.genomeinformatics.org
aws s3 ls --no-sign-request --endpoint-url $EP s3://icgc25k-open/
aws s3 ls --no-sign-request --endpoint-url $EP s3://icgc25k-open/release_28/ --recursive | grep -i PRAD-UK
aws s3 cp --no-sign-request --endpoint-url $EP s3://icgc25k-open/release_28/<path_from_ls> .
```

I haven't listed the bucket, so take the exact file paths from the `ls` output.

#### 2. Open raw reads today: GIAB HG008, a pancreatic cancer (PDAC) tumour/normal set

This is the only broadly consented, fully open cancer tumour + normal-tissue WGS set I know of, and it's in your own field. Per the [Scientific Data paper](https://www.nature.com/articles/s41597-025-05438-2):
- **Tumour:** primary PDAC, grade **G2**, ypT2 **N1**. It isn't distantly metastatic, but it is node-positive, and the patient had neoadjuvant therapy.
- **Samples:**

| Sample | What it is | Role in your design |
|---|---|---|
| HG008-T | Tumour **cell line** | T |
| HG008-N-P | Normal pancreas tissue | Adjacent / border |
| HG008-N-D | Normal duodenum tissue | Germline normal (no blood sample was usable) |

- **Data:** Illumina, Element, Ultima, PacBio HiFi, ONT and Hi-C, all public on the NCBI GIAB FTP.

```bash
BASE=https://ftp.ncbi.nlm.nih.gov/ReferenceSamples/giab/data_somatic/HG008/

# 1) list the tree without downloading anything
wget --spider -r -l 6 --no-parent -nv "$BASE" 2>&1 | grep -oE 'https://[^ ]+' | sort -u > HG008_urls.txt
grep -Ei 'fastq|\.bam|\.cram' HG008_urls.txt | less

# 2) download only what you choose (e.g., one Illumina run per sample)
wget -c -x -nH --cut-dirs=3 -i selected_urls.txt
```

Caveats:
- The tumour is a cell line, so there's no microenvironment and no stromal dilution.
- There's only one patient, so it's a benchmark and a way to test the pipeline end to end, not a cohort.
- For the pipeline, map T → HG008-T, B → N-P, N → N-D, and swap the prostate driver list for KRAS, TP53, CDKN2A and SMAD4.

A practical order would be: get the pipeline running on HG008 now, and check PRAD-UK grades in the open ICGC tables while the DACO application is under review.

Sources:
- [Legacy ICGC 25K data access](https://docs.icgc-argo.org/docs/data-access/icgc-25k-data)
- [Applying for DACO access](https://docs.icgc-argo.org/docs/data-access/daco/applying)
- [DACO approval process](https://docs.icgc-argo.org/docs/data-access/daco/approval)
- [HG008, Scientific Data 2025](https://www.nature.com/articles/s41597-025-05438-2)
- [GIAB HG008 FTP](https://ftp.ncbi.nlm.nih.gov/ReferenceSamples/giab/data_somatic/HG008/)
- [NIST Cancer Genome in a Bottle](https://www.nist.gov/programs-projects/cancer-genome-bottle)

The command works. It was just slow because `--recursive` on `release_28/` lists all 86 projects before `grep` filters them. List only the project prefix and it comes back in seconds:

```bash
EP=https://object.genomeinformatics.org
B=s3://icgc25k-open/release_28
S3="aws s3 --no-sign-request --endpoint-url $EP"

$S3 ls $B/data/PRAD-UK/                  # one folder per donor (about 226)
$S3 ls $B/headers/                       # the part-*.gz files have no header line; column names are here
```

**Download only the clinical tables** (a few hundred KB in total):
```bash
$S3 cp $B/headers/ headers/ --recursive
$S3 sync $B/data/PRAD-UK/ PRAD-UK/ --exclude "*" \
    --include "*/donor/*" --include "*/specimen/*" --include "*/sample/*"

for t in donor specimen sample; do
  zcat PRAD-UK/*/$t/part-*.gz > PRAD-UK.$t.tsv
done
```

**Filter to Gleason grade group 2–3** in Python. Check the header file names from the `ls` output first; the glob below assumes each file name contains the table name.

```python
import glob
import pandas as pd

def load(t):
    hdr = open(glob.glob(f"headers/*{t}*")[0]).readline().rstrip("\n").split("\t")
    return pd.read_csv(f"PRAD-UK.{t}.tsv", sep="\t", names=hdr, dtype=str)

donor, spec, samp = load("donor"), load("specimen"), load("sample")

def isup(g):
    """Gleason text (e.g. '3+4', '4+3=7') -> ISUP grade group."""
    try:
        a, b = (int(x) for x in str(g).split("=")[0].split("+"))
    except ValueError:
        return None
    s = a + b
    return 1 if s <= 6 else (2 if (a, b) == (3, 4) else 3 if (a, b) == (4, 3) else 4 if s == 8 else 5)

tum = spec[spec.specimen_type.str.contains("tumour", case=False, na=False)].copy()
tum["isup_gg"] = tum.tumour_grade.map(isup)
print(tum[["icgc_donor_id", "specimen_type", "tumour_grade", "tumour_stage"]].drop_duplicates().head(20))
print(tum.isup_gg.value_counts(dropna=False))
```

Things to check once you have the tables:
- **Look at how `tumour_grade` is actually written before trusting `isup()`.** It may be "3+4", a total like "7", or free text. Check `tum.tumour_grade.unique()` and adjust the parser.
- **M status may not be recorded.** It could be in `tumour_stage` (TNM) or `donor_tumour_stage_at_diagnosis`. If it's missing everywhere, radical prostatectomy with no recorded metastasis is the usual proxy for M0.
- **PRAD-UK is larger than the EGA field-effect set:** about 226 donors against 30 + 7 men. To link the two, the EGA sample descriptions (e.g. `PD9170b`, a Sanger sample ID) should match `submitted_sample_id` in the `sample` table. That tells you which eligible donors are also in EGAD00001004125.
- **The `ssm_open` folders hold each donor's open somatic mutation calls.** You can run signatures or a quick mutation-burden check on them before you ever get BAM access.

### **Filter to Gleason grade group 2–3** in Python.

Check the header file names from the `ls` output first; the glob below assumes each file name contains the table name.


In [7]:
root_data

PosixPath('/home/flavio/uv/mutation_analysis/data')

In [8]:
root_icgc = root_data / 'ICGC'
root_icgc

PosixPath('/home/flavio/uv/mutation_analysis/data/ICGC')

In [12]:
str(root_icgc) + f"/headers/"

'/home/flavio/uv/mutation_analysis/data/ICGC/headers/'

In [25]:
import glob, gzip
import pandas as pd
from pathlib import Path

prad = root_icgc / "PRAD-UK"

def open_text(p: Path):
    """Open plain or gzipped text transparently."""
    with open(p, "rb") as fh:
        is_gz = fh.read(2) == b"\x1f\x8b"
    return gzip.open(p, "rt") if is_gz else open(p)

def header_file(t: str) -> Path:
    """Header for table t: exact name match, so 'donor' != 'donor_exposure'."""
    hits = [p for p in (root_icgc / "headers").rglob("*") if p.is_file()]
    exact = [p for p in hits if p.name.split(".")[0] == t or p.parent.name == t]
    if len(exact) != 1:
        raise FileNotFoundError(f"{t}: {exact or hits}")
    return exact[0]

def load(t: str) -> pd.DataFrame:
    with open_text(header_file(t)) as fh:
        hdr = fh.readline().rstrip("\n").split("\t")
    parts = sorted(prad.glob(f"*/{t}/part-*.gz"))
    return pd.concat((pd.read_csv(p, sep="\t", names=hdr, dtype=str) for p in parts),
                     ignore_index=True).drop_duplicates()

donor, spec, samp = load("donor"), load("specimen"), load("sample")

def isup(g):
    """Gleason text (e.g. '3+4', '4+3=7') -> ISUP grade group."""
    try:
        a, b = (int(x) for x in str(g).split("=")[0].split("+"))
    except ValueError:
        return None
    s = a + b
    return 1 if s <= 6 else (2 if (a, b) == (3, 4) else 3 if (a, b) == (4, 3) else 4 if s == 8 else 5)

df_tum = spec[spec.specimen_type.str.contains("tumour", case=False, na=False)].copy()
df_tum["isup_gg"] = df_tum.tumour_grade.map(isup)

fname = 'prostate_cancer_Cruk_aws_all_samples.tsv'
print(df_tum.shape)
pdwritecsv(df_tum, fname, root_icgc)

(279, 30)


True

In [20]:
df_tum.head(3).T

,1,3,5
icgc_specimen_id,SP125579,SP125584,SP131541
project_code,PRAD-UK,PRAD-UK,PRAD-UK
study_specimen_involved_in,PCAWG,PCAWG,NaN
submitted_specimen_id,0056_CRUK_PC_0056_T1,0086_CRUK_PC_0086_T1,0027_CRUK_PC_0027_T1
icgc_donor_id,DO221503,DO221504,DO223959
submitted_donor_id,0056_CRUK_PC_0056,0086_CRUK_PC_0086,0027_CRUK_PC_0027
specimen_type,Primary tumour - solid tissue,Primary tumour - solid tissue,Primary tumour - solid tissue
specimen_type_other,NaN,NaN,NaN
specimen_interval,72,125,126
specimen_donor_treatment_type,no treatment,no treatment,no treatment


In [18]:
df_tum[["icgc_donor_id", "specimen_type", "tumour_grade", "tumour_stage"]].drop_duplicates().head(20)

,icgc_donor_id,specimen_type,tumour_grade,tumour_stage
1,DO221503,Primary tumour - solid tissue,3+3,T3aN0Mx
3,DO221504,Primary tumour - solid tissue,3+3,T2cNxMx
5,DO223959,Primary tumour - solid tissue,3+3,T3aN0Mx
7,DO223960,Primary tumour - solid tissue,3+3,T2cN0Mx
8,DO223961,Primary tumour - solid tissue,4+5,T4xNxMx
11,DO223962,Primary tumour - solid tissue,5+4,T3xNxMx
13,DO223963,Primary tumour - solid tissue,5+4,T4xNxMx
14,DO223964,Primary tumour - solid tissue,4+3,T3aN0Mx
16,DO223965,Primary tumour - solid tissue,3+3,T3aN0Mx
19,DO223966,Primary tumour - solid tissue,4+4,T3aN0Mx


It loaded. There are three things to check before you trust that file:

**1. Nothing has been filtered yet.** `df_tum` contains every tumour specimen, and `isup_gg` is only an added column. As written, the file named `gleason_2_3` holds all grades.

**2. Check that `isup()` understood the grade format.** If `tumour_grade` isn't written like "3+4", every value becomes `None`:
```python
print(df_tum.tumour_grade.value_counts(dropna=False).head(20))
print(df_tum.isup_gg.value_counts(dropna=False))
print(df_tum.filter(regex="stage|grad|metast").columns.tolist())   # where TNM / M status lives
```

**3. Check which donors have the full T / border / blood set.** `submitted_specimen_id` uses the same naming as the EGA sample titles (`0056_CRUK_PC_0056_T1` here, `0002_CRUK_PC_0002_Blood` on EGA), so you can link the two directly. Build the availability table from `spec` (all specimen types), not `df_tum`:
```python
print(spec.specimen_type.value_counts())

def arm(t: str) -> str:
    t = t.lower()
    if "blood" in t:
        return "N_blood"
    if "normal" in t:
        return "B_normal_tissue"      # adjacent or other normal prostate tissue
    return "T"

spec["arm"] = spec.specimen_type.fillna("").map(arm)
avail = (spec.groupby(["icgc_donor_id", "arm"]).size().unstack(fill_value=0))
gg = df_tum.groupby("icgc_donor_id").isup_gg.max()   # highest grade across tumour regions
avail = avail.join(gg)

eligible = avail[(avail.T > 0) & (avail.get("B_normal_tissue", 0) > 0)
                 & (avail.get("N_blood", 0) > 0) & avail.isup_gg.isin([2, 3])]
print(len(avail), "donors;", len(eligible), "with T + normal tissue + blood and ISUP 2-3")

df_sel = spec[spec.icgc_donor_id.isin(eligible.index)]
pdwritecsv(df_sel, "prostate_cancer_Cruk_aws_gleason_2_3.tsv", root_icgc)
```

Notes on the choices here:
- **The grade is the highest across a donor's tumour regions.** Patients with multifocal tumours are graded by their worst lesion, which matches clinical practice.
- **If `B_normal_tissue` is 0 for almost everyone,** the morphologically normal samples may be in the EGA field-effect dataset and not in the ICGC specimen table. In that case, match the EGA sample titles against `submitted_specimen_id` instead.
- **The 7 cancer-free men won't be in PRAD-UK**, because ICGC donors are cancer patients. They exist only in EGA.
- **M0 isn't enforced yet.** Once the third print in step 2 shows which column holds M status, add it to the `eligible` filter.

In [23]:
df_tum.isup_gg.value_counts(dropna=False)

isup_gg
2.0    124
NaN     61
3.0     43
4.0     20
1.0     18
5.0     13
Name: count, dtype: int64

In [24]:
df_tum.tumour_grade.value_counts(dropna=False).head(20)

tumour_grade
3+4    124
NaN     61
4+3     43
3+3     18
4+4     18
4+5      8
5+4      4
3+5      2
5+5      1
Name: count, dtype: int64

In [22]:
df_tum.filter(regex="stage|grad|metast").columns   # where TNM / M status lives

Index(['tumour_grading_system', 'tumour_grade', 'tumour_grade_supplemental', 'tumour_stage_system',
       'tumour_stage', 'tumour_stage_supplemental'],
      dtype='object')

The parser works, and the grade groups add up correctly:
- **Grade group 4 (Gleason 8):** 18 × 4+4 plus 2 × 3+5 = 20
- **Grade group 5 (Gleason 9–10):** 8 + 4 + 1 = 13

**167 tumour specimens are grade group 2 or 3** (124 × 3+4 and 43 × 4+3), about 70% of those with a grade. These are specimens, not donors: a man with several tumour regions counts more than once.

Two gaps remain: the 61 specimens without a grade, and M status. This checks both:

Once I see how `tumour_stage` is written (e.g. `T2cN0M0` or split fields), I'll add the M0 condition. Then run the availability code from the previous message to count the men who are grade group 2–3, M0, and have tumour, normal tissue and blood.

In [28]:
# 1) do the 61 ungraded specimens have the grade in the supplemental column?
print(df_tum.loc[df_tum.tumour_grade.isna(), "tumour_grade_supplemental"].value_counts(dropna=False).head(10))
print()

# 2) where TNM / metastasis is recorded
print(df_tum[["tumour_stage_system", "tumour_stage", "tumour_stage_supplemental"]]
      .value_counts(dropna=False).head(20))
print()
print(donor.filter(regex="stage|metast|relapse|status|interval").columns.tolist())
print()
print(donor.donor_tumour_stage_at_diagnosis.value_counts(dropna=False).head(15))
print()

# 3) grade group per donor = worst region; how many men are 2-3
gg = df_tum.groupby("icgc_donor_id").isup_gg.max()
print(gg.value_counts(dropna=False).sort_index())

tumour_grade_supplemental
NaN    61
Name: count, dtype: int64

tumour_stage_system  tumour_stage  tumour_stage_supplemental
pTNM                 T3aN0Mx       NaN                          71
NaN                  NaN           NaN                          51
pTNM                 T2cNxMx       NaN                          41
                     T3aNxMx       NaN                          37
                     T2cN0Mx       NaN                          20
                     T3bN0Mx       NaN                          11
                     TxxNxMx       NaN                          10
                     T3bN1Mx       NaN                           7
                     T3bNxMx       NaN                           6
                     T2cN0M0       NaN                           6
                     T2aNxMx       NaN                           4
                     T3aN0M0       NaN                           4
                     T3aN1Mx       NaN                           3
     

At this point the M criterion is really a decision about what "non-metastatic" should mean, because **M is almost never recorded**. Only about 12 specimens have an explicit M0, one has M1, and everything else is **Mx** (not assessed).

That's normal for a prostatectomy cohort. Pathologists stage the resected organ, **so pathological M is rarely assigned**. Radical prostatectomy is offered only to men who are clinically non-metastatic, which in practice makes Mx at surgery equivalent to cM0. Requiring an explicit M0 would leave you with about 10 men for no biological reason.

The rule I'd suggest is **"no evidence of M1"**. The single M1 case (`T2cN0M1`) is excluded, and so is any distant relapse recorded at or near diagnosis. N status is a separate question, so it's a switch:



**Estimated numbers**, from the donor-level grades: 117 + 43 = **about 160 men with grade group 2–3**. Removing the M1 case, and the pN1 men if you set `REQUIRE_N0 = True` (roughly a dozen specimens are N1), should still leave well over 100. The script prints the exact count.

**Look at `donor_relapse_type` in the output.** A later "distant" relapse means the patient was non-metastatic at diagnosis but progressed. Whether to keep those men depends on the question:
- **For field-effect biology,** keep them.
- **For a strictly indolent cohort,** drop them.

**The next filter is sample availability.** These are ICGC tumour donors, most of them probably tumour + blood only. The morphologically normal tissue (your "border" arm) exists for the 30 men in the EGA field-effect dataset. Intersecting `sel` with those men, via `submitted_specimen_id` ↔ EGA sample titles such as `0056_CRUK_PC_0056`, will show how many have all three arms.


#### forget about non-metastatic

In [ ]:
import re

def tnm(s):
    """'T3bN1Mx' -> ('3b', '1', 'x'); None if unparseable."""
    m = re.match(r"T(\w+?)N(\w)M(\w)$", str(s))
    return m.groups() if m else None

stage = df_tum.assign(
    pT=df_tum.tumour_stage.map(lambda s: (tnm(s) or (None,) * 3)[0]),
    pN=df_tum.tumour_stage.map(lambda s: (tnm(s) or (None,) * 3)[1]),
    pM=df_tum.tumour_stage.map(lambda s: (tnm(s) or (None,) * 3)[2]),
)
per_donor = stage.groupby("icgc_donor_id").agg(
    isup_gg=("isup_gg", "max"),
    pN=("pN", lambda x: "1" if (x == "1").any() else ("0" if (x == "0").any() else "x")),
    pM1=("pM", lambda x: (x == "1").any()),
)

per_donor = (df_tum.groupby("icgc_donor_id").isup_gg.max()   # worst tumour region per man
             .to_frame())
sel = per_donor[per_donor.isup_gg.isin([2, 3])]
print(len(per_donor), "donors ->", len(sel), "ISUP grade group 2-3")

df_sel = spec[spec.icgc_donor_id.isin(sel.index)]            # all their specimens (T, normal, blood)
df_sel.reset_index(inplace=True, drop=True)
print(df_sel.shape)
pdwritecsv(df_sel, "prostate_cancer_Cruk_aws_gleason_2_3.tsv", root_icgc)

df_sel.head(6)

225 donors -> 160 ISUP grade group 2-3
(334, 29)


,icgc_specimen_id,project_code,study_specimen_involved_in,submitted_specimen_id,icgc_donor_id,submitted_donor_id,specimen_type,specimen_type_other,specimen_interval,specimen_donor_treatment_type,...,tumour_histological_type,tumour_grading_system,tumour_grade,tumour_grade_supplemental,tumour_stage_system,tumour_stage,tumour_stage_supplemental,digital_image_of_stained_section,percentage_cellularity,level_of_cellularity
14,SP131551,PRAD-UK,NaN,0051_CRUK_PC_0051_T1,DO223964,0051_CRUK_PC_0051,Primary tumour - solid tissue,NaN,365,no treatment,...,8140/3,Gleason,4+3,NaN,pTNM,T3aN0Mx,NaN,NaN,53,NaN
15,SP131552,PRAD-UK,NaN,0051_CRUK_PC_0051_Blood,DO223964,0051_CRUK_PC_0051,Normal - blood derived,NaN,483,no treatment,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,SP131558,PRAD-UK,NaN,0054_CRUK_PC_0054_Blood,DO223967,0054_CRUK_PC_0054,Normal - blood derived,NaN,56,no treatment,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21,SP131557,PRAD-UK,NaN,0054_CRUK_PC_0054_T1,DO223967,0054_CRUK_PC_0054,Primary tumour - solid tissue,NaN,56,no treatment,...,8140/3,Gleason,3+4,NaN,pTNM,T3aN0Mx,NaN,NaN,48,NaN
22,SP203352,PRAD-UK,NaN,0054_CRUK_PC_0054_N,DO223967,0054_CRUK_PC_0054,Normal - tissue adjacent to primary,NaN,56,no treatment,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25,SP131562,PRAD-UK,NaN,0057_CRUK_PC_0057_Blood,DO223969,0057_CRUK_PC_0057,Normal - blood derived,NaN,-31,no treatment,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
